# cell 0 — 安装 WHL（Serverless）

In [0]:
import sys
from pathlib import Path

_local_src = str(Path.cwd() / "src")
if _local_src not in sys.path:
    sys.path.insert(0, _local_src)

from lakehouse.transforms.macro_builder import build_gold_market_macro_daily

mkt_df = spark.table("enterprise.silver_market.crypto_ohlc_1d")
fx_df = spark.table("enterprise.silver_ref.ecb_fx_daily")
fed_df = spark.table("enterprise.silver_ref.fred_series_daily")

# Build the table using the extracted function with a 5-day safe TTL limit
gold_macro_df = build_gold_market_macro_daily(mkt_df, fx_df, fed_df, max_fill_days=5)

gold_macro_df.write.mode("overwrite").format("delta").partitionBy("trade_date").saveAsTable("enterprise.gold_ref.market_macro_daily")
